In [1]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q sentence-transformers

In [2]:
import os

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

/tmp/ipykernel_14049/1590039611.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [3]:
loader = PyPDFDirectoryLoader("./corpus")

documents = loader.load()

print(f"{len(documents)} documentos cargados.")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)


248 documentos cargados.


In [ ]:
os.environ["GOOGLE_API_KEY"] = "xxxxxxxxxxxxxxxx"

In [5]:
print(f"{len(documents)} documentos cargados.\n")

for i, doc in enumerate(documents[:10]):
    print("=" * 80)
    print(f"Documento {i+1}")
    print("SOURCE:")
    print(doc.metadata.get("source"))
    print("\nPrimeros 300 caracteres:")
    print(doc.page_content[:300])
    print()

248 documentos cargados.

Documento 1
SOURCE:
corpus/EU_AI-Act-overview-30-May-2024.pdf.pdf

Primeros 300 caracteres:
Future of Life Institute | Available online: artificialintelligenceact.eu/high-level-summary 
High-level summary of the AI Act 
In this article we provide you with a high-level summary of the AI Act, 
selecting the parts which are most likely to be relevant to you regardless 
of who you are. We prov

Documento 2
SOURCE:
corpus/EU_AI-Act-overview-30-May-2024.pdf.pdf

Primeros 300 caracteres:
Prohibited AI systems (Chapter II, Art. 5) 
AI systems:  
• deploying subliminal, manipulative, or deceptive techniques to distort behaviour and impair 
informed decision-making, causing significant harm. 
• exploiting vulnerabilities related to age, disability, or socio-economic circumstances to di

Documento 3
SOURCE:
corpus/EU_AI-Act-overview-30-May-2024.pdf.pdf

Primeros 300 caracteres:
High risk AI systems (Chapter III) 
Classification rules for high-risk AI systems (Art. 6) 
Hi

In [6]:
print("Dividiendo documentos...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"{len(chunks)} chunks creados.")

Dividiendo documentos...
813 chunks creados.


In [7]:
import time
print("Inicio carga embeddings:", time.strftime("%H:%M:%S"))

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Modelo cargado:", time.strftime("%H:%M:%S"))

Inicio carga embeddings: 21:24:21


/tmp/ipykernel_14049/3643073580.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo cargado: 21:24:24


In [8]:
print("Construyendo índice FAISS...")
print("Cantidad de chunks:", len(chunks))

print("Primer chunk:")
print(chunks[0].page_content[:200])

texto_prueba = [chunks[0].page_content]

emb = embeddings.embed_documents(texto_prueba)

print("Embeddings devueltos:")
print(emb)

print("Cantidad:", len(emb))
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

Construyendo índice FAISS...
Cantidad de chunks: 813
Primer chunk:
Future of Life Institute | Available online: artificialintelligenceact.eu/high-level-summary 
High-level summary of the AI Act 
In this article we provide you with a high-level summary of the AI Act, 
Embeddings devueltos:
[[-0.05546236038208008, -0.032262172549963, 0.002179254312068224, -0.04198870435357094, 0.08395953476428986, -0.0015582729829475284, 0.01235120091587305, 0.03902916982769966, 0.0106412498280406, 0.026169035583734512, -0.012207282707095146, -0.015398446470499039, 0.014270061627030373, 0.022132884711027145, -0.027865521609783173, 0.015622404403984547, 0.004849723074585199, -0.07774654030799866, -0.08956368267536163, 0.030945386737585068, 0.06807548552751541, -0.014881961047649384, 0.025938184931874275, -0.002106461441144347, -0.06900782138109207, -0.024651896208524704, 0.012876653112471104, -0.08592340350151062, -0.05049268901348114, 0.036605168133974075, 0.02022555097937584, 0.030884595587849617, 0.052

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

In [10]:
import requests

url = "https://raw.githubusercontent.com/CelesteBox/Lunes_9AM/main/docs/system_prompt.md"

response = requests.get(url)

system_prompt = response.text

print(system_prompt[:500])


```

# 1 Identidad:

Eres "Lunes 9 a.m.", un asistente especializado en gobernanza práctica de Inteligencia Artificial. Tu función es ayudar a equipos técnicos a incorporar criterios de gobernanza, transparencia y responsabilidad antes, durante y después del despliegue de sistemas de IA.

---

# 2 Objetivo:

Transformar principios y marcos internacionales de gobernanza de IA en decisiones técnicas concretas, trazables y defendibles.

---

# 3 Sobre cómo responder. Siempre dar:

a) Formato y est


In [11]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [12]:
def manual_rag(query: str):

    # Recuperar documentos
    docs = retriever.invoke(query)

    print("\nDOCUMENTOS RECUPERADOS\n")

    for i, doc in enumerate(docs):
        print(f"\n--- Documento {i+1} ---")
        print("SOURCE:", doc.metadata.get("source"))
        print("PAGE:", doc.metadata.get("page"))
        print(doc.page_content[:500])

    # Construir contexto
    contexto = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # Prompt con system prompt integrado
    prompt = f"""
{system_prompt}

====================
CONTEXTO
====================

{contexto}

====================
PREGUNTA
====================

{query}

====================
RESPUESTA
====================
"""

    respuesta = llm.invoke(prompt)

    return respuesta.content

In [ ]:
print("\n=========================================")
print("Asistente listo.")
print("Escribí 'salir' para terminar.")
print("=========================================\n")

while True:

    pregunta = input("Pregunta: ")

    if pregunta.lower() == "salir":
        print("Hasta luego.")
        break

    respuesta = manual_rag(pregunta)

    print("\nRespuesta:\n")
    print(respuesta)
    print("\n" + "=" * 80 + "\n")


Asistente listo.
Escribí 'salir' para terminar.



Pregunta:  



DOCUMENTOS RECUPERADOS


--- Documento 1 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 11
12

--- Documento 2 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 7
8

--- Documento 3 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 23
24

--- Documento 4 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 15
16

--- Documento 5 ---
SOURCE: corpus/NIST_AI_RMF_Playbook.pdf.pdf
PAGE: 36
MANAGE

--- Documento 6 ---
SOURCE: corpus/NIST_AI_RMF_Playbook.pdf.pdf
PAGE: 4
GOVERN

--- Documento 7 ---
SOURCE: corpus/NIST_AI_RMF_Playbook.pdf.pdf
PAGE: 60
MAP

--- Documento 8 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 12
13
II. 
Fines y 
objetivos

--- Documento 9 ---
SOURCE: corpus/NIST_AI_RMF_Playbook.pdf.pdf
PAGE: 1
MANAGE........................................................................................................................................................................ 35 
MANAGE 1.1 ........................................................

Pregunta:  que principios recomienda la OCDE para la gobernanza de la IA?



DOCUMENTOS RECUPERADOS


--- Documento 1 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 17
o en situación de vulnerabilidad, incluidos, entre 
otros, los niños, las personas de edad, las personas 
con discapacidad o los enfermos. En el marco de 
esas interacciones, las personas nunca deberían ser 
cosificadas, su dignidad no debería ser menoscabada 
de ninguna otra manera, y sus derechos humanos y 
libertades fundamentales nunca deberían ser objeto de 
violación o abusos.

--- Documento 2 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 21
mecanismos de gobernanza adecuados, protegidos por 
los sistemas judiciales y aplicados a lo largo del ciclo de 
vida de los sistemas de IA. Los marcos de protección 
de datos y todo mecanismo conexo deberían tomar 
como referencia los principios y normas internacionales 
de protección de datos relativos a la recopilación, la 
utilización y la divulgación de datos personales y al 
ejercicio de sus derechos por parte de los interesado

Pregunta:  ¿Qué principios recomienda la OECD para la gobernanza de inteligencia artificial?



DOCUMENTOS RECUPERADOS


--- Documento 1 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 2
Adoptada el 23 de noviembre de 2021
Recomendación sobre 
la ética de  
la inteligencia 
artificial

--- Documento 2 ---
SOURCE: corpus/OECD_AI_Principles_ES.pdf.pdf
PAGE: 0
OECD  AI  Principles  
Metadatos  del  Documento  ●  Fuente  oficial:  https://oecd.ai/en/ai-principles ●  Fecha  de  consulta:  27/06/2026  
Resumen  de  los  Principios  de  la  OCDE  
Los  Principios  de  la  OCDE  sobre  IA  promueven  el  uso  de  una  inteligencia  artificial  
innovadora
 
y
 
confiable
 
que
 
respete
 
los
 
derechos
 
humanos
 
y
 
los
 
valores
 
democráticos.
 
Adoptados
 
originalmente
 
en
 
mayo
 
de
 
2019
 
y
 
actualizados
 
en
 
mayo
 
de
 
2024,
 
establecen


--- Documento 3 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 0
Recomendación sobre   
la ética de 
la inteligencia 
artificial
Adoptada el 23 de noviembre de 2021

--- Documento 4 ---
SOURCE: corpus/UNESCO_Ethics_o

In [37]:
manual_rag("¿Qué riesgos debería documentar un equipo antes de desplegar un sistema de inteligencia artificial?")


DOCUMENTOS RECUPERADOS


--- Documento 1 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 9
enfoque:
a) los sistemas de IA son tecnologías de 
procesamiento de la información que integran 
modelos y algoritmos que producen una 
capacidad para aprender y realizar tareas 
cognitivas, dando lugar a resultados como 
la predicción y la adopción de decisiones en 
entornos materiales y virtuales. Los sistemas de 
IA están diseñados para funcionar con diferentes 
grados de autonomía, mediante la modelización y 
representación del conocimiento y la explotación 
de datos y el cálculo de correlac

--- Documento 2 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 35
niveles educativos, a fin de dar a los trabajadores actuales 
y a las nuevas generaciones una oportunidad equitativa 
de encontrar empleo en un mercado en rápida evolución 
y para asegurar que sean conscientes de los aspectos 
éticos de los sistemas de IA. Junto a las competencias

--- Documento 3 ---
SOURCE: corpus/UNESC

'Un equipo debería documentar los riesgos relacionados con:\n\n*   **Información errónea, desinformación y discurso de odio**, así como los **daños causados por el uso indebido de los datos personales**.\n*   Las **repercusiones de los sistemas de IA en la cultura**, especialmente de las aplicaciones de procesamiento del lenguaje natural (PLN), como la traducción automática y los asistentes de voz, en los matices del lenguaje y la expresión humanos.\n*   El **impacto de los sistemas de IA en los derechos humanos**, incluidos los derechos de los niños, y sus repercusiones.\n*   Reforzar o perpetuar **aplicaciones y resultados discriminatorios o sesgados**, para garantizar la equidad de dichos sistemas.\n*   La posibilidad de que los sistemas causen **daños indebidos o muestren comportamientos no deseados**.'

In [44]:
manual_rag("¿Qué documentación debería existir antes de desplegar un sistema de IA de alto impacto?")


DOCUMENTOS RECUPERADOS


--- Documento 1 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 40
41
133. A fin de promover las mejores políticas y prácticas 
relacionadas con la ética de la IA, deberían elaborarse 
instrumentos e indicadores adecuados para evaluar su 
eficacia y eficiencia en función de normas, prioridades y 
objetivos acordados, incluidos objetivos específicos para 
las personas pertenecientes a poblaciones desfavorecidas 
y marginadas y personas vulnerables o en situación de 
vulnerabilidad, así como el impacto de los sistemas de 
IA en los planos individual y social. El 

--- Documento 2 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 25
el impacto de los sistemas de IA en el respeto de 
los derechos humanos, el estado de derecho y las 
sociedades inclusivas. Los Estados Miembros deberían 
también poder evaluar los efectos socioeconómicos 
de los sistemas de IA en la pobreza y velar por que la 
brecha entre los ricos y los pobres, así como la brecha 
dig

'Antes de desplegar un sistema de IA de alto impacto, debería existir la siguiente documentación y procesos:\n\n*   **Evaluaciones del impacto ético de los sistemas de IA**: Estas deben realizarse para anticipar las repercusiones, atenuar los riesgos, evitar las consecuencias perjudiciales, facilitar la participación de los ciudadanos y hacer frente a los desafíos sociales (Párrafo 53).\n*   **Mecanismos de supervisión adecuados**: La evaluación debe establecer la auditabilidad, la trazabilidad y la explicabilidad, que permitan evaluar los algoritmos, los datos y los procesos de concepción (Párrafo 53).\n*   **Examen externo de los sistemas**: La evaluación debe incluir un examen externo de los sistemas (Párrafo 53).\n*   **Evaluaciones del impacto en la privacidad**: Estas evaluaciones deben incluir consideraciones sociales y éticas de su utilización y un empleo innovador del enfoque de privacidad desde la etapa de concepción (Párrafo 34).\n*   **Evaluaciones de los aspectos éticos de

In [47]:
manual_rag("¿Cuál es la fecha exacta en que Argentina aprobará una ley nacional de inteligencia artificial?")


DOCUMENTOS RECUPERADOS


--- Documento 1 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 21
los regímenes nacionales e internacionales pertinentes 
en materia de responsabilidad funcionen eficazmente. 
La falta de transparencia también podría mermar la 
posibilidad de impugnar eficazmente las decisiones 
basadas en resultados producidos por los sistemas de 
IA y, por lo tanto, podría vulnerar el derecho a un juicio 
imparcial y a un recurso efectivo, y limita los ámbitos en 
los que estos sistemas pueden utilizarse legalmente.
38. Si bien hay que hacer todo lo posible por aumentar la 


--- Documento 2 ---
SOURCE: corpus/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 32
plural a ellas y, en particular, para garantizar que la 
recomendación algorítmica aumente la notoriedad de 
los contenidos locales y la posibilidad de descubrirlos.
99. Los Estados Miembros deberían promover nuevas 
investigaciones en la intersección entre la IA y la 
propiedad intelectual, por ejemplo, para determinar 

'La información no está disponible en los documentos consultados.'